# 03: 一次元輝度分布 → 時空図・ピーク位置確率分布・ヒートマップ（Fig. 1 低解像度パート）

`data/luminance_1d/` の一次元輝度分布CSV（01の出力）から、論文Fig. 1の各パネルおよびFig. S1/S2/S6を生成するノートブック。

**処理の流れ**
1. パネル定義（どのサンプル・チャンバーをどの図に使うか）を読み、必要なCSVを一括読み込み
2. 前処理: 80 mm＝0.08 mm刻み（1000点）への線形補間 → 列ごとの最小値を背景として減算 → 左右端5点をゼロ化
3. ピーク検出: 移動平均（窓5）→ 輝度閾値70超の連続領域ごとに最大値位置をピークとして抽出、mm換算
4. パネルごとに確率分布・時空図・ヒートマップを描画し、`outputs/Fig1/` にSVG保存

**使い方**: パス設定を確認して Run All。個別パネルだけ再実行も可（各パネルセルは独立）。

**注意**: Fig. 1C/F の時空図は高解像度データ（02ノートブックの出力）から作成するため、
本ノートブックでは末尾のプレースホルダとし、02完成後に追加する。

## セットアップ

In [ ]:
import json
from pathlib import Path
from typing import Union, Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

In [ ]:
# ============ パス設定 ============
DATA_ROOT = Path("../data/luminance_1d")     # 01の出力（一次元輝度分布CSV）
OUT_DIR = Path("../outputs/Fig1")            # 図の出力先
CONFIG_PATH = Path("../config/samples_lowres.json")

OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH, encoding="utf-8") as f:
    config = json.load(f)
samples = config["samples"]


def csv_path(sample_id: str, chamber: str) -> Path:
    """sample_id と chamber から新命名規則のCSVパスを返す。"""
    s = samples[sample_id]
    stem = Path(s["video_file"]).stem
    return DATA_ROOT / s["date"] / f"lumi1d_{s['date']}_{stem}_{chamber}.csv" 

## パネル定義

**どのサンプル・チャンバーをどの図に使うか**をここに集約する（旧Integrationノートブックの各セルに
散在していた `date_l` 等の選択を、確認済みの構成で一元化したもの）。
- Fig.1D: (0,0,0) 3動画×3チャンバー=9（1117を含む構成で確定。旧cell140は0320+1208のみだったが修正）
- Fig.1G: (1,5,1) 9データセット（num_dence 3動画のbottomを含む構成で確定）
- Fig.1H: 第2段は「Pp=1/5」として扱う（ファイル名の0.25表記との齟齬は要確認のまま）

In [ ]:
TMB = ["top", "middle", "bottom"]

def pc(sample_id, chambers):
    return [(sample_id, ch) for ch in chambers]

# ---- 確率分布パネル（各要素: データセットのリストと柱条件） ----
PANELS = {
    "Fig1D_000": {
        "datasets": pc("250320_149A3445", TMB) + pc("251117_149A3606", TMB) + pc("251208_149A3642", TMB),
        "Wp": 0, "Np": 1, "Pp": 1/2,
    },
    "Fig1G_151": {
        "datasets": pc("250320_149A3444", TMB)
                    + [("250807_149A3575", "bottom"), ("250807_149A3576", "bottom"), ("250807_149A3577", "bottom")]
                    + pc("251117_149A3607", TMB),
        "Wp": 1, "Np": 1, "Pp": 1/2,
    },
    # ---- Fig.1H ヒートマップの各行 ----
    "H_Pp80_2": {"datasets": None,  # Fig1G_151と同一（下で参照）
                 "Wp": 1, "Np": 1, "Pp": 1/2, "markers": [80/2]},
    "H_Pp80_3": {
        "datasets": pc("251208_149A3653", TMB) + pc("250807_149A3578", TMB) + pc("251208_149A3639", TMB),
        "Wp": 1, "Np": 1, "Pp": 1/3, "markers": [80/3],
    },
    "H_Pp80_5": {
        "datasets": pc("251117_149A3608", TMB),
        "Wp": 1, "Np": 1, "Pp": 1/5, "markers": [80/5],
    },
    "H_Np2": {
        "datasets": [("250320_149A3448", "bottom"), ("250617_149A3481", "bottom")],
        "Wp": 1, "Np": 2, "Pp": 1/2, "markers": [80/3, 80/3*2],
    },
    "H_Np3": {
        "datasets": [("250320_149A3448", "middle"), ("250617_149A3481", "middle")],
        "Wp": 1, "Np": 3, "Pp": 1/2, "markers": [80/4, 80/4*2, 80/4*3],
    },
    "H_Np5": {
        "datasets": [("250320_149A3448", "top"), ("250617_149A3481", "top")],
        "Wp": 1, "Np": 5, "Pp": 1/2, "markers": [80/6*i for i in range(1, 6)],
    },
    # ---- Fig.S6（Hp系列）: 確率分布＋時空図 ----
    "S6_Hp1": {
        "datasets": [("250320_149A3449", "bottom"), ("250617_149A3480", "bottom")],
        "Wp": 1, "Np": 1, "Pp": 1/2,
    },
    "S6_Hp2.5": {
        "datasets": [("250320_149A3446", "bottom"), ("250617_149A3480", "middle")],
        "Wp": 1, "Np": 1, "Pp": 1/2,
    },
    # ---- Wp系確率分布（論文中の使い先は要確認。データとしては保持） ----
    "Wp_dist": {
        "datasets": pc("250320_149A3447", TMB) + pc("250617_149A3479", TMB),
        "Wp": None, "Np": 1, "Pp": 1/2,
    },
}
PANELS["H_Pp80_2"]["datasets"] = PANELS["Fig1G_151"]["datasets"]

# ---- 全サンプル時空図（Fig.S1 / S2） ----
KYMO_PANELS = {
    "FigS1_000": PANELS["Fig1D_000"] | {},
    "FigS2_151": PANELS["Fig1G_151"] | {},
}

# ピーク検出パラメータ（論文Methods対応）
THRESHOLD_LUMINANCE = config["defaults"]["luminance_threshold_downstream"]  # 70
LIMITED_REGION_RATIO_DIST = 8/8   # 確率分布: 全域
LIMITED_REGIONS_CENTER = 4/8
LIMITED_REGION_RATIO_KYMO = 3/8   # 時空図: 中央30mm（x=25〜55mm）

## 関数群

いずれも旧Integrationノートブックからの移植（処理内容は同一）。

In [ ]:
def resample_to_physical_space(
    luminance_data: Union[Sequence[float], np.ndarray],
    physical_length_mm: float = 80.0,
    resolution_mm: float = 0.08
) -> np.ndarray:
    """
    物理長（mm）基準で輝度分布を補間する。
    ロットごとに輝度分布長（pixel数）が揃わないため、全て80mm長である前提で
    0.08mmごと（1000点）の共通物理グリッドに線形補間で載せ替える。
    """
    original_pixel_count = len(luminance_data)
    physical_x_original = np.linspace(0, physical_length_mm, original_pixel_count)
    physical_x_target = np.arange(0, physical_length_mm, resolution_mm)
    interpolator = interp1d(physical_x_original, luminance_data, kind='linear',
                            bounds_error=False, fill_value="extrapolate")
    return interpolator(physical_x_target)


def subtract_background(lumi_df):
    """各位置（列）ごとに全フレーム中の最小輝度値をバックグラウンドとみなし、減算する。"""
    background = lumi_df.min(axis=0)
    return lumi_df - background


def get_peak_position(
        limited_region_ratio,
        limited_regions_center,
        luminance_values_use,
        threshold_luminance,
        Wp,
        fig_tag=False,
):
    """一次元輝度分布の時系列から下降プルームのピーク位置[mm]を抽出する。

    手順: 注目領域の切り出し → 移動平均（窓5）による平滑化 →
          閾値(threshold_luminance)以上の連続領域を検出し、各領域内の最大値位置をピークとする。
    """
    len_of_pixel = len(luminance_values_use[0])

    rimited_rigion_dists = []
    for i in range(len(luminance_values_use)):
        left_boundary = int(len_of_pixel * (limited_regions_center - limited_region_ratio * 1/2))
        right_boundary = int(len_of_pixel * (limited_regions_center + limited_region_ratio * 1/2))
        rimited_rigion_dists.append(luminance_values_use[i][left_boundary:right_boundary])

    window_size = 5  # 移動平均をとるデータの数（細かいピークを消す）

    def moving_average(data, window_size):
        return np.convolve(data, np.ones(window_size) / window_size, mode='same')

    smoothed_luminances = [moving_average(d, window_size) for d in rimited_rigion_dists]

    all_peak_positions = []
    left_side_pix_of_rimited_rigion = (len_of_pixel * (1 - limited_region_ratio)) / 2

    for luminance_data in smoothed_luminances:
        luminance_data = np.array(luminance_data)
        mask = luminance_data >= threshold_luminance

        regions = []
        start = None
        for i in range(len(mask)):
            if mask[i] and start is None:
                start = i
            elif not mask[i] and start is not None:
                regions.append((start, i - 1))
                start = None
        if start is not None:
            regions.append((start, len(mask) - 1))

        for start, end in regions:
            peak_idx = start + np.argmax(luminance_data[start:end + 1]) + left_side_pix_of_rimited_rigion
            all_peak_positions.append(peak_idx)

    px_to_mm = 80 * limited_region_ratio / len(rimited_rigion_dists[0])
    peak_positions_mm = np.array(all_peak_positions) * px_to_mm
    return px_to_mm, peak_positions_mm

In [ ]:
def make_whole_figure(
    limited_region_ratio, px_to_mm, Wp, Np, Pp,
    luminance_values_use, peak_positions_mm, grapth_titles,
    ax1_tag, ax2_tag, ax3_tag, bins_of_ax3, ylim_max_of_ax3,
    fig_save_path,
):
    """時空図(ax1)・時間平均輝度分布(ax2)・ピーク位置ヒストグラム(ax3)を選択的に描画しSVG保存する。"""
    plot_flags = [ax1_tag, ax2_tag, ax3_tag]
    selected_indices = [i for i, flag in enumerate(plot_flags) if flag]
    n_axes = len(selected_indices)
    if n_axes == 0:
        print("何も表示対象が選ばれていません。")
        return

    height_ratios_dict = {0: 1.5, 1: 1, 2: 1}
    selected_height_ratios = [height_ratios_dict[i] for i in selected_indices]
    fig, axes = plt.subplots(n_axes, 1, figsize=(12, 6 * n_axes), sharex=True,
                             gridspec_kw={'height_ratios': selected_height_ratios})
    if n_axes == 1:
        axes = [axes]

    # 柱位置の垂直線
    if Pp == 1/2:
        pillar_posi = [80 / (Np + 1) * i for i in range(1, Np + 1)]
    else:
        pillar_posi = [80 * Pp for i in range(1, Np + 1)]
    vlines_to_use = []
    for pos in pillar_posi:
        vlines_to_use.append(pos - Wp / 2)
        vlines_to_use.append(pos + Wp / 2)

    current_ax = 0
    left_limit = int((len(luminance_values_use[0]) * (1 - limited_region_ratio)) / 2)
    right_limit = int((len(luminance_values_use[0]) * (1 + limited_region_ratio)) / 2)
    luminance_values_use = [lum[left_limit:right_limit] for lum in luminance_values_use]

    if ax1_tag:  # 時空図
        ax = axes[current_ax]
        flipped_luminance = np.flipud(luminance_values_use)
        im = ax.imshow(flipped_luminance, aspect='auto', cmap='gray', origin='upper',
                       extent=[25, 55, 0, 30], vmin=0, vmax=255)
        for vline in vlines_to_use:
            ax.axvline(x=vline, color='red', linestyle='--', linewidth=0.5)
        ax.tick_params(axis='y', labelsize=25)
        ax.tick_params(axis='x', labelsize=25)
        current_ax += 1

    if ax2_tag:  # 時間平均輝度分布
        ax = axes[current_ax]
        luminance_values_mean = np.mean(luminance_values_use, axis=0)
        x_values = np.linspace(0, 80, len(luminance_values_mean))
        ax.plot(x_values, luminance_values_mean, label=f"pillar width={Wp}mm",
                color="#FF7F50", linestyle="-")
        for vline in vlines_to_use:
            ax.axvline(x=vline, color="red", linestyle="--", linewidth=0.5)
        ax.set_xlim(0, 80)
        ax.set_ylim(0, 255)
        ax.set_ylabel('Luminance (0-255)', fontsize=25)
        ax.tick_params(axis='y', labelsize=20)
        ax.legend(loc="upper left", fontsize=12)
        ax.grid(True, linestyle="--", alpha=0.6)
        ax.set_title(grapth_titles[1], fontsize=22)
        current_ax += 1

    if ax3_tag:  # ピーク位置ヒストグラム
        ax = axes[current_ax]
        ll = (len(luminance_values_use[0]) * (1 - limited_region_ratio)) / 2
        rl = (len(luminance_values_use[0]) * (1 + limited_region_ratio)) / 2
        if len(peak_positions_mm) > 0:
            bins = (np.arange(ll * px_to_mm, rl * px_to_mm, px_to_mm * 1)
                    if bins_of_ax3 is None else bins_of_ax3)
            ax.hist(peak_positions_mm, bins=bins,
                    weights=np.ones_like(peak_positions_mm) * (100 / len(peak_positions_mm)),
                    alpha=0.75, color="black", edgecolor="black")
            ax.set_xlabel('X (mm)', fontsize=35)
            ax.set_ylabel('probability\ndensity(%)', fontsize=30)
            ax.grid(axis='y', linestyle='--', alpha=0.7)
            ax.set_xlim(25, 55)
            ax.set_ylim(0, ylim_max_of_ax3)
            ax.tick_params(axis='x', labelsize=30)
            ax.tick_params(axis='y', labelsize=30)
            for vline in vlines_to_use:
                ax.axvline(x=vline, color="red", linestyle="--", linewidth=0.5)
        else:
            ax.text(0.5, 0.5, "有効なピークが見つかりませんでした。閾値を調整してください。",
                    horizontalalignment='center', verticalalignment='center',
                    transform=ax.transAxes, fontsize=14)
            ax.axis('off')

    fig.savefig(fig_save_path, format="svg", bbox_inches="tight")
    plt.tight_layout()
    plt.show()


def plot_peak_heatmap(peak_lists, marker_x_lists, row_labels, save_path,
                      x_min=0, x_max=80, bin_width=1.0, vmax=0.05):
    """複数系列のピーク位置分布を1行ずつのヒートマップで表示する（Fig.1H形式）。"""
    bins = np.arange(x_min, x_max + bin_width, bin_width)
    heatmap_rows = []
    for peak_posi_mm in peak_lists:
        counts, _ = np.histogram(peak_posi_mm, bins=bins)
        heatmap_rows.append(counts / counts.sum() if counts.sum() > 0 else counts)
    heatmap_data = np.array(heatmap_rows)

    n_series = len(peak_lists)
    fig, ax = plt.subplots(figsize=(8, 0.55 * n_series + 1.2))
    row_height, row_gap = 0.5, 0.2
    row_pitch = row_height + row_gap

    for i in range(n_series):
        y_bottom = i * row_pitch
        y_top = y_bottom + row_height
        im = ax.imshow(heatmap_data[i][np.newaxis, :], aspect="auto", cmap="inferno",
                       extent=[x_min, x_max, y_top, y_bottom], origin="upper",
                       vmin=0, vmax=vmax)

    for i, x_pillar_list in enumerate(marker_x_lists):
        for x_pillar in x_pillar_list:
            ax.text(x_pillar, i * row_pitch, "▼", color="black", fontsize=11,
                    ha="center", va="bottom")

    ax.set_xlabel("x [mm]", fontsize=14)
    ax.set_xlim(x_min, x_max)
    ax.set_xticks(np.arange(x_min, x_max + 1, 10))
    ytick_positions = [i * row_pitch + row_height / 2 for i in range(n_series)]
    ax.set_yticks(ytick_positions)
    ax.set_yticklabels(row_labels, fontsize=12)
    ax.set_ylim(n_series * row_pitch - row_gap, 0)
    for side in ["left", "right", "top"]:
        ax.spines[side].set_visible(False)
    ax.tick_params(axis="y", length=0)
    fig.colorbar(im, ax=ax, label="Probability")
    fig.savefig(save_path, format="svg", bbox_inches="tight")
    plt.show()

## データ読み込み・前処理

パネル定義で参照される全 (sample, chamber) のCSVを読み込み、
1000点規格化 → 背景除去 → 左右端5点ゼロ化 を適用して `lumi_pre` 辞書に格納する。

In [ ]:
# パネルから必要な (sample_id, chamber) を集約
needed = set()
for p in PANELS.values():
    needed.update(p["datasets"])

lumi_pre = {}
for sid, ch in sorted(needed):
    path = csv_path(sid, ch)
    assert path.exists(), f"CSVが見つかりません: {path}（01の実行漏れ？）"
    df = pd.read_csv(path, index_col=0)

    # 前処理0: 1000点規格化（80mm / 0.08mm）
    arr = np.vstack([resample_to_physical_space(row) for row in df.values])
    dfp = pd.DataFrame(arr)

    # 前処理1: 背景除去 / 前処理2: 左右端5点を0に
    dfp = subtract_background(dfp)
    dfp.iloc[:, :5] = 0
    dfp.iloc[:, -5:] = 0

    lumi_pre[(sid, ch)] = dfp
    print(f"{sid} [{ch}] {df.shape} -> {dfp.shape}")

print(f"\n読み込み完了: {len(lumi_pre)} データセット")

In [ ]:
def integrate_peaks(panel, fig_tag=False):
    """パネル定義に従い全データセットのピーク位置[mm]を統合して返す。"""
    peak_all, memo = [], []
    px_to_mm = None
    for sid, ch in panel["datasets"]:
        lum = lumi_pre[(sid, ch)].values.tolist()
        px_to_mm, peaks = get_peak_position(
            LIMITED_REGION_RATIO_DIST, LIMITED_REGIONS_CENTER,
            lum, THRESHOLD_LUMINANCE, panel["Wp"], fig_tag)
        peak_all.extend(float(x) for x in peaks)
        memo.append(f"{sid} [{ch}] peak数: {len(peaks)}")
    for m in memo:
        print(m)
    print(f"peak位置検出数の合計: {len(peak_all)}")
    return px_to_mm, np.array(peak_all)

## Fig. 1D | (0,0,0) 下降プルーム位置の確率分布

3動画（250320_149A3445, 251117_149A3606, 251208_149A3642）× 3チャンバー = N=9。

In [ ]:
panel = PANELS["Fig1D_000"]
px_to_mm, peaks_1D = integrate_peaks(panel)
ref = lumi_pre[panel["datasets"][0]].values.tolist()
make_whole_figure(LIMITED_REGION_RATIO_DIST, px_to_mm,
                  panel["Wp"], panel["Np"], panel["Pp"],
                  ref, peaks_1D, ["", "", ""],
                  ax1_tag=False, ax2_tag=False, ax3_tag=True,
                  bins_of_ax3=170, ylim_max_of_ax3=5,
                  fig_save_path=OUT_DIR / "Fig1D_000_distribution.svg")

## Fig. 1G | (1,5,1) 下降プルーム位置の確率分布

250320_149A3444×3 + num_dence 3動画のbottom + 251117_149A3607×3 = N=9。

In [ ]:
panel = PANELS["Fig1G_151"]
px_to_mm, peaks_1G = integrate_peaks(panel)
ref = lumi_pre[panel["datasets"][0]].values.tolist()
make_whole_figure(LIMITED_REGION_RATIO_DIST, px_to_mm,
                  panel["Wp"], panel["Np"], panel["Pp"],
                  ref, peaks_1G, ["", "", ""],
                  ax1_tag=False, ax2_tag=False, ax3_tag=True,
                  bins_of_ax3=170, ylim_max_of_ax3=5,
                  fig_save_path=OUT_DIR / "Fig1G_151_distribution.svg")

## Fig. 1H | 柱位置・本数を変化させたときの確率分布ヒートマップ

行構成（上から）: Pp=80/2, Pp=80/3, Pp=80/5, Np=2 (Pp=80/3·i), Np=3 (80/4·i), Np=5 (80/6·i)。
※旧コードは80/2行を含む6行構成。論文Fig.1Hが5行（80/2行なし）の場合は `HEATMAP_ROWS` から
`"H_Pp80_2"` を外すだけで対応可能。

In [ ]:
HEATMAP_ROWS = ["H_Pp80_2", "H_Pp80_3", "H_Pp80_5", "H_Np2", "H_Np3", "H_Np5"]
ROW_LABELS = ["Pp=80/2", "Pp=80/3", "Pp=80/5", "Np=2", "Np=3", "Np=5"]

peak_lists, marker_lists = [], []
for key in HEATMAP_ROWS:
    panel = PANELS[key]
    print(f"===== {key} =====")
    _, peaks = integrate_peaks(panel)
    peak_lists.append(peaks)
    marker_lists.append(panel["markers"])

plot_peak_heatmap(peak_lists, marker_lists, ROW_LABELS,
                  save_path=OUT_DIR / "Fig1H_heatmap.svg")

## Fig. S1 | (0,0,0) 全サンプル時空図（N=9）

中央30 mm（x=25〜55 mm）の時空図を、データセットごとに個別SVGとして出力する。

In [ ]:
kymo_dir = OUT_DIR / "FigS1_all_kymographs"
kymo_dir.mkdir(exist_ok=True)
panel = PANELS["Fig1D_000"]
for sid, ch in panel["datasets"]:
    lum = lumi_pre[(sid, ch)].values.tolist()
    px_to_mm, peaks = get_peak_position(LIMITED_REGION_RATIO_KYMO, LIMITED_REGIONS_CENTER,
                                        lum, THRESHOLD_LUMINANCE, panel["Wp"])
    print(f"{sid} [{ch}]")
    make_whole_figure(LIMITED_REGION_RATIO_KYMO, px_to_mm,
                      panel["Wp"], panel["Np"], panel["Pp"],
                      lum, peaks, ["", "", ""],
                      ax1_tag=True, ax2_tag=False, ax3_tag=False,
                      bins_of_ax3=170, ylim_max_of_ax3=2,
                      fig_save_path=kymo_dir / f"kymo_{sid}_{ch}.svg")

## Fig. S2 | (1,5,1) 全サンプル時空図（N=9）

In [ ]:
kymo_dir = OUT_DIR / "FigS2_all_kymographs"
kymo_dir.mkdir(exist_ok=True)
panel = PANELS["Fig1G_151"]
for sid, ch in panel["datasets"]:
    lum = lumi_pre[(sid, ch)].values.tolist()
    px_to_mm, peaks = get_peak_position(LIMITED_REGION_RATIO_KYMO, LIMITED_REGIONS_CENTER,
                                        lum, THRESHOLD_LUMINANCE, panel["Wp"])
    print(f"{sid} [{ch}]")
    make_whole_figure(LIMITED_REGION_RATIO_KYMO, px_to_mm,
                      panel["Wp"], panel["Np"], panel["Pp"],
                      lum, peaks, ["", "", ""],
                      ax1_tag=True, ax2_tag=False, ax3_tag=False,
                      bins_of_ax3=170, ylim_max_of_ax3=2,
                      fig_save_path=kymo_dir / f"kymo_{sid}_{ch}.svg")

## Fig. S6 | 柱高さ Hp を変化させたときの時空図と確率分布

Hp=1 mm（A,B,C）とHp=2.5 mm（D,E,F）。各高さにつき2サンプル（0320・0617ロット）。

In [ ]:
for key, tag in [("S6_Hp1", "Hp1"), ("S6_Hp2.5", "Hp2p5")]:
    panel = PANELS[key]
    kymo_dir = OUT_DIR / f"FigS6_{tag}"
    kymo_dir.mkdir(exist_ok=True)
    print(f"===== {key} =====")

    # A/B, D/E: 各サンプルの時空図
    for sid, ch in panel["datasets"]:
        lum = lumi_pre[(sid, ch)].values.tolist()
        px_to_mm, peaks = get_peak_position(LIMITED_REGION_RATIO_KYMO, LIMITED_REGIONS_CENTER,
                                            lum, THRESHOLD_LUMINANCE, panel["Wp"])
        make_whole_figure(LIMITED_REGION_RATIO_KYMO, px_to_mm,
                          panel["Wp"], panel["Np"], panel["Pp"],
                          lum, peaks, ["", "", ""],
                          ax1_tag=True, ax2_tag=False, ax3_tag=False,
                          bins_of_ax3=170, ylim_max_of_ax3=2,
                          fig_save_path=kymo_dir / f"kymo_{sid}_{ch}.svg")

    # C/F: 統合確率分布
    px_to_mm, peaks = integrate_peaks(panel)
    ref = lumi_pre[panel["datasets"][0]].values.tolist()
    make_whole_figure(LIMITED_REGION_RATIO_DIST, px_to_mm,
                      panel["Wp"], panel["Np"], panel["Pp"],
                      ref, peaks, ["", "", ""],
                      ax1_tag=False, ax2_tag=False, ax3_tag=True,
                      bins_of_ax3=170, ylim_max_of_ax3=5,
                      fig_save_path=kymo_dir / f"distribution_{tag}.svg")

## （参考）Wp = 0.5, 2, 5 の統合確率分布

旧Integrationに存在した解析。論文中の使用先が未確定のため参考出力（要確認）。

In [ ]:
panel = PANELS["Wp_dist"]
for sid, ch in panel["datasets"]:
    wp = samples[sid]["chambers"][ch]["condition"]
    lum = lumi_pre[(sid, ch)].values.tolist()
    wp_val = float(wp.split("=")[1]) if "Wp=" in wp else 1
    px_to_mm, peaks = get_peak_position(LIMITED_REGION_RATIO_DIST, LIMITED_REGIONS_CENTER,
                                        lum, THRESHOLD_LUMINANCE, wp_val)
    print(f"{sid} [{ch}] {wp}: peak数 {len(peaks)}")

---
## 【プレースホルダ】Fig. 1C / 1F 時空図（高解像度データ）

Fig. 1C（(0,0,0)）およびFig. 1F（(1,5,1)）の時空図は、251208の高解像度動画から
02ノートブックで抽出する一次元輝度分布を入力とする。02完成後にこのセクションへ追加する。
対象データ（予定）: `data/luminance_1d_highres/` 配下のCSV。Fig.4実験パネル・Fig.S5（水深系列）も同様。